# ELK Homework

This notebook captures the local Elasticsearch, Kibana, and Logstash setup for the homework.
It includes the Logstash pipeline, the Python async logger, a shell health check, and the Kibana URLs to submit.

## Submission Links

- Kibana console for the `shakespeare` dataset: `http://10.3.134.62:5601/app/dev_tools#/console?_g=()`
- Kibana console for the top-10 word aggregation on `python-logstash-homework-*`: `http://10.3.134.62:5601/app/dev_tools#/console?_g=()`

Run the Dev Tools queries below in Kibana to show the homework datasets.

In [1]:
from pathlib import Path
ROOT = Path('/home/hadoop/homework/elk')
ROOT

PosixPath('/home/hadoop/homework/elk')

## Logstash Pipeline

```conf
input {
  tcp {
    port => 5959
    codec => json
  }
}

filter {
  if [message] !~ /^python-logstash:/ {
    drop { }
  }

  if [extra][top_word] {
    mutate {
      add_field => { "top_word" => "%{[extra][top_word]}" }
    }
  }

  if [extra][sentence] {
    mutate {
      add_field => { "sentence" => "%{[extra][sentence]}" }
    }
  }

  if [top_word] {
    mutate {
      lowercase => [ "top_word" ]
    }
  }
}

output {
  elasticsearch {
    hosts => ["http://127.0.0.1:9200"]
    index => "python-logstash-homework-%{+YYYY.MM.dd}"
  }

  stdout {
    codec => rubydebug
  }
}
```

## Python Async Logger

```python
from __future__ import annotations

import logging
import random
import time
from pathlib import Path

from logstash_async.handler import AsynchronousLogstashHandler


WORDS = [
    "spark",
    "elastic",
    "kibana",
    "logstash",
    "python",
    "cluster",
    "dashboard",
    "search",
    "index",
    "query",
]

SENTENCE_TEMPLATES = [
    "streaming data improves {word} analysis",
    "the {word} pipeline stays healthy",
    "students use {word} for this homework",
    "the {word} dashboard highlights trends",
    "{word} search helps debug events",
]


def build_logger(database_path: Path) -> logging.Logger:
    logger = logging.getLogger("python-logstash-homework")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(
        AsynchronousLogstashHandler(
            host="127.0.0.1",
            port=5959,
            database_path=str(database_path),
        )
    )
    return logger


def emit_messages(logger: logging.Logger, total_messages: int = 120) -> None:
    random.seed(42)
    for idx in range(total_messages):
        word = random.choices(
            WORDS,
            weights=[14, 11, 7, 10, 8, 5, 4, 6, 9, 3],
            k=1,
        )[0]
        sentence = random.choice(SENTENCE_TEMPLATES).format(word=word)
        level = random.choice([logging.INFO, logging.WARNING, logging.ERROR])

        # Roughly one quarter of events should be filtered out by Logstash.
        prefix = "python-filebeat" if idx % 4 == 0 else "python-logstash"
        logger.log(
            level,
            f"{prefix}: {sentence}",
            extra={"top_word": word, "sentence": sentence},
        )
        time.sleep(0.02)


def close_logger(logger: logging.Logger) -> None:
    for handler in logger.handlers:
        handler.flush()
        handler.close()


def main() -> None:
    logger = build_logger(Path("logstash_events.db"))
    try:
        emit_messages(logger)
    finally:
        close_logger(logger)


if __name__ == "__main__":
    main()
```

## Elasticsearch Health Script

```bash
#!/usr/bin/env bash
set -euo pipefail

echo "=== cluster health ==="
curl -sS http://127.0.0.1:9200/_cluster/health?pretty
echo
echo "=== nodes ==="
curl -sS http://127.0.0.1:9200/_cat/nodes?v
echo
echo "=== homework indices ==="
curl -sS http://127.0.0.1:9200/_cat/indices/python-logstash-homework-*?v
```

## Cluster Health Output

In [2]:
!bash /home/hadoop/homework/elk/check_elastic_health.sh

=== cluster health ===
{
  "cluster_name" : "elasticsearch",
  "status" : "yellow",
  "timed_out" : false,
  "number_of_nodes" : 1,
  "number_of_data_nodes" : 1,
  "active_primary_shards" : 14,
  "active_shards" : 14,
  "relocating_shards" : 0,
  "initializing_shards" : 0,
  "unassigned_shards" : 7,
  "delayed_unassigned_shards" : 0,
  "number_of_pending_tasks" : 0,
  "number_of_in_flight_fetch" : 0,
  "task_max_waiting_in_queue_millis" : 0,
  "active_shards_percent_as_number" : 66.66666666666666
}

=== nodes ===
ip        heap.percent ram.percent cpu load_1m load_5m load_15m node.role   master name
127.0.0.1           27          83  -1    0.07    0.14     0.16 cdfhilmrstw *      bigdata

=== homework indices ===
health status index                               uuid                   pri rep docs.count docs.deleted store.size pri.store.size
yellow open   python-logstash-homework-2026.03.17 SRwMhHlpR8egLhv-Ad4V2w   1   1         90            0     99.1kb         99.1kb


## Homework Queries For Kibana Dev Tools

In [3]:
import json

shakespeare_query = {
    'query': {'match': {'play_name': 'Hamlet'}},
    'size': 3,
}
top_words_query = {
    'size': 0,
    'aggs': {
        'top_words': {
            'terms': {
                'field': 'top_word.keyword',
                'size': 10,
                'order': {'_count': 'desc'},
            }
        }
    },
}
print('GET shakespeare/_search')
print(json.dumps(shakespeare_query, indent=2))
print()
print('GET python-logstash-homework-*/_search')
print(json.dumps(top_words_query, indent=2))

GET shakespeare/_search
{
  "query": {
    "match": {
      "play_name": "Hamlet"
    }
  },
  "size": 3
}

GET python-logstash-homework-*/_search
{
  "size": 0,
  "aggs": {
    "top_words": {
      "terms": {
        "field": "top_word.keyword",
        "size": 10,
        "order": {
          "_count": "desc"
        }
      }
    }
  }
}


## Current Counts From Elasticsearch

In [4]:
import json
import subprocess

def curl_json(url: str, method: str = 'GET', payload: dict | None = None):
    cmd = ['curl', '-sS', '-X', method, url]
    if payload is not None:
        cmd.extend(['-H', 'Content-Type: application/json', '-d', json.dumps(payload)])
    out = subprocess.check_output(cmd, text=True)
    return json.loads(out)

cluster_health = curl_json('http://127.0.0.1:9200/_cluster/health')
shakespeare_count = curl_json('http://127.0.0.1:9200/shakespeare/_count')
top_words = curl_json(
    'http://127.0.0.1:9200/python-logstash-homework-*/_search',
    method='GET',
    payload={
        'size': 0,
        'aggs': {
            'top_words': {
                'terms': {
                    'field': 'top_word.keyword',
                    'size': 10,
                }
            }
        },
    },
)
cluster_health, shakespeare_count, top_words['aggregations']['top_words']['buckets']

({'cluster_name': 'elasticsearch',
  'status': 'yellow',
  'timed_out': False,
  'number_of_nodes': 1,
  'number_of_data_nodes': 1,
  'active_primary_shards': 14,
  'active_shards': 14,
  'relocating_shards': 0,
  'initializing_shards': 0,
  'unassigned_shards': 7,
  'delayed_unassigned_shards': 0,
  'number_of_pending_tasks': 0,
  'number_of_in_flight_fetch': 0,
  'task_max_waiting_in_queue_millis': 0,
  'active_shards_percent_as_number': 66.66666666666666},
 {'count': 111396,
  '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}},
 [{'key': 'elastic', 'doc_count': 16},
  {'key': 'logstash', 'doc_count': 15},
  {'key': 'spark', 'doc_count': 14},
  {'key': 'index', 'doc_count': 13},
  {'key': 'cluster', 'doc_count': 10},
  {'key': 'python', 'doc_count': 7},
  {'key': 'dashboard', 'doc_count': 6},
  {'key': 'kibana', 'doc_count': 4},
  {'key': 'query', 'doc_count': 3},
  {'key': 'search', 'doc_count': 2}])